# Week 2, Monday: Data Preprocessing, Train/Test Split & Cross-Validation
Dataset: **Titanic** (predict whether a passenger survived). It is messy on purpose: it has missing values and text categories, so it is perfect for practising preprocessing.

Run top to bottom. Cells marked 🧪 are exercises for you. Write answers in your own words in the Markdown cells below them.

## 0. Setup

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
sns.set_theme(style='whitegrid')
np.random.seed(42)

## 1. Load data and explore (EDA)

In [ ]:
df = sns.load_dataset('titanic')
print(df.shape)
display(df.head())
df.info()

In [ ]:
# Missing values: how many, and what percentage?
missing = df.isnull().sum()
pd.DataFrame({'missing': missing, 'percent': (missing/len(df)*100).round(1)}).query('missing > 0')

🧪 **Exercise 1:** Which column has the most missing values? What percentage? Would you fill it or drop it, and why?

*Your answer:* 

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=df, x='sex', y='survived', ax=ax[0]); ax[0].set_title('Survival rate by sex')
sns.barplot(data=df, x='pclass', y='survived', ax=ax[1]); ax[1].set_title('Survival rate by class')
sns.histplot(data=df, x='age', hue='survived', bins=30, ax=ax[2]); ax[2].set_title('Age distribution by survival')
plt.tight_layout(); plt.show()
print('Overall survival rate:', round(df['survived'].mean(), 3))

🧪 **Exercise 2:** What do the three plots tell you? Which groups were most likely to survive?

*Your answer:* 

## 2. Choose features and split the data
- We **drop `deck`**: about 77% missing, too much to fill reliably.
- We **drop `class`, `who`, `adult_male`, `alive`, `embark_town`, `alone`**: they duplicate other columns (or `alive` is the target in disguise, which would be **leakage**).
- Features kept: numeric `age, fare, sibsp, parch, pclass`; categorical `sex, embarked`.

In [ ]:
num_cols = ['age', 'fare', 'sibsp', 'parch', 'pclass']
cat_cols = ['sex', 'embarked']
X = df[num_cols + cat_cols]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print('Survival rate  train:', round(y_train.mean(), 3), ' test:', round(y_test.mean(), 3))

**Why `stratify=y`?** It keeps the survival rate about the same in the train and test sets, so the split is fair.

**The golden rule:** split **first**, then learn anything (averages, categories, scaling) from the **training set only**. If you compute them on the full dataset, information from the test set leaks into training and your test score becomes too optimistic. This is called **data leakage**.

## 3. Handle missing data (step by step, fit on TRAIN only)

In [ ]:
# Numeric: fill missing age with the MEDIAN of the training set
num_imputer = SimpleImputer(strategy='median').fit(X_train[num_cols])
print('Medians learned from TRAIN:', dict(zip(num_cols, num_imputer.statistics_.round(2))))
print('Median age on full data   :', df['age'].median(), '(slightly different: that is the leakage we avoid)')

# Categorical: fill missing embarked with the MOST FREQUENT value of the training set
cat_imputer = SimpleImputer(strategy='most_frequent').fit(X_train[cat_cols])
print('Most frequent categories from TRAIN:', cat_imputer.statistics_)

In [ ]:
Xtr_num = pd.DataFrame(num_imputer.transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
Xte_num = pd.DataFrame(num_imputer.transform(X_test[num_cols]),  columns=num_cols, index=X_test.index)
Xtr_cat = pd.DataFrame(cat_imputer.transform(X_train[cat_cols]), columns=cat_cols, index=X_train.index)
Xte_cat = pd.DataFrame(cat_imputer.transform(X_test[cat_cols]),  columns=cat_cols, index=X_test.index)
print('Missing left in train:', Xtr_num.isnull().sum().sum() + Xtr_cat.isnull().sum().sum())

🧪 **Exercise 3:** Why do we use the **median** for age instead of the mean? (Hint: look at the age histogram and think about outliers.) Also, when might dropping rows with missing values be better than filling them?

*Your answer:* 

## 4. Encode categories and scale numbers
- **One-hot encoding:** models need numbers, not text. `sex` becomes columns like `sex_female`, `sex_male` with 0/1 values.
- **Scaling:** `fare` is in the hundreds while `sibsp` is 0-8. Scaling puts features on a similar scale (mean 0, standard deviation 1). Models like kNN and logistic regression need it; trees don't.

In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(Xtr_cat)
Xtr_oh = pd.DataFrame(ohe.transform(Xtr_cat), columns=ohe.get_feature_names_out(cat_cols), index=Xtr_cat.index)
Xte_oh = pd.DataFrame(ohe.transform(Xte_cat), columns=ohe.get_feature_names_out(cat_cols), index=Xte_cat.index)
display(Xtr_oh.head())

scaler = StandardScaler().fit(Xtr_num)
Xtr_sc = pd.DataFrame(scaler.transform(Xtr_num), columns=num_cols, index=Xtr_num.index)
Xte_sc = pd.DataFrame(scaler.transform(Xte_num), columns=num_cols, index=Xte_num.index)
print('Before scaling:'); display(Xtr_num.describe().loc[['mean','std']].round(2))
print('After scaling :'); display(Xtr_sc.describe().loc[['mean','std']].round(2))

In [ ]:
# Put the pieces back together
Xtr_final = pd.concat([Xtr_sc, Xtr_oh], axis=1)
Xte_final = pd.concat([Xte_sc, Xte_oh], axis=1)
print(Xtr_final.shape); display(Xtr_final.head())

🧪 **Exercise 4:** Why do we call `.fit()` on the training data only, but `.transform()` on both train and test? What would go wrong if we fitted the scaler on the test set too?

*Your answer:* 

## 5. Baseline model and a single train/test score

In [ ]:
model = LogisticRegression(max_iter=1000).fit(Xtr_final, y_train)
print('Train accuracy:', round(model.score(Xtr_final, y_train), 3))
print('Test accuracy :', round(model.score(Xte_final, y_test), 3))

Also compare with the **majority-class baseline**: always predicting 'did not survive'. Your model should beat this.

In [ ]:
print('Baseline (always predict the majority class):', round(1 - y_test.mean(), 3))

## 6. Why one split is not enough: cross-validation
A single split gives one score, and it depends on *which* rows happened to land in the test set. Try different random splits and watch the score move.

In [ ]:
# Same preprocessing + model wrapped in a Pipeline (we study Pipelines properly on Wednesday).
# A Pipeline refits the imputer, encoder and scaler on each training fold, which avoids leakage.
preprocess = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])
pipe = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=1000))])

# 1) Different single splits give different scores
scores = []
for seed in range(10):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    scores.append(pipe.fit(Xa, ya).score(Xb, yb))
print('10 different splits:', np.round(scores, 3))
print('min =', round(min(scores), 3), ' max =', round(max(scores), 3))

In [ ]:
# 2) 5-fold cross-validation: every row is used for testing exactly once
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X, y, cv=cv)
print('5 fold scores:', np.round(cv_scores, 3))
print(f'Mean = {cv_scores.mean():.3f}  Std = {cv_scores.std():.3f}')
plt.bar(range(1, 6), cv_scores); plt.axhline(cv_scores.mean(), color='r', ls='--', label='mean')
plt.ylim(0.6, 0.9); plt.xlabel('fold'); plt.ylabel('accuracy'); plt.legend(); plt.show()

**How 5-fold CV works:** the data is cut into 5 parts. Each part takes one turn as the test set while the other 4 train the model. You get 5 scores, and the **mean** is a more reliable estimate than a single split, while the **standard deviation** shows how stable it is.

🧪 **Exercise 5:** Look at the spread of the 10 single-split scores versus the 5 CV scores. In your own words, why is the CV mean a more trustworthy estimate than one split?

*Your answer:* 

In [ ]:
# 🧪 Exercise 6: change the imputation of age from 'median' to 'mean' in the pipeline, or drop 'embarked'
# from cat_cols, rebuild the pipeline, and rerun the CV cell. Does the CV mean change noticeably?

## 7. Summary notes (fill in, in your own words)
**Data leakage:** 

**Why cross-validation is more reliable than one split:** 

**Imputing vs dropping missing data:** 

**One-hot vs ordinal encoding (when would you use each?):** 

**Which models need scaling and which don't:** 
